### Jacky

In [1]:
import json
import subprocess
import sys
import urllib.request
import urllib.error


def check_github_connection(timeout=10):
    """检查是否能够访问 GitHub API，并返回结果字典。"""
    url = "https://api.github.com"  # GitHub API 根地址
    result = {
        "reachable": False,
        "status_code": None,
        "reason": None,
        "remote_url": None,
        "git_remote_exists": False,
        "git_remote_url": None,
    }

    try:
        with urllib.request.urlopen(url, timeout=timeout) as response:
            result["reachable"] = True
            result["status_code"] = response.getcode()  # HTTP 状态码
            result["reason"] = response.reason  # HTTP 原因短语
    except urllib.error.HTTPError as exc:
        result["status_code"] = exc.code
        result["reason"] = str(exc)
    except urllib.error.URLError as exc:
        result["reason"] = str(exc)
    except Exception as exc:
        result["reason"] = str(exc)

    # 尝试读取本地 Git 远程地址，适用于当前工作目录有 git 仓库时
    try:
        completed = subprocess.run(
            [sys.executable, "-c", "import subprocess; import json; print(json.dumps(subprocess.check_output(['git','remote','-v'], text=True)))"],
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        if completed.returncode == 0 and completed.stdout:
            remote_text = json.loads(completed.stdout)
            result["git_remote_exists"] = True
            result["git_remote_url"] = remote_text.strip()  # 可能包含多条远程地址信息
            if "github.com" in result["git_remote_url"]:
                result["remote_url"] = result["git_remote_url"]
    except Exception:
        pass

    return result


if __name__ == "__main__":
    status = check_github_connection()
    print(json.dumps(status, indent=2, ensure_ascii=False))


{
  "reachable": true,
  "status_code": 200,
  "reason": "OK",
  "remote_url": null,
  "git_remote_exists": false,
  "git_remote_url": null
}
